# 01 青空文庫 — データ取得・前処理・Word2Vec学習

**このノートブックの目的**

1. 青空文庫から孤独関連テキストを収集する
2. MeCab（fugashi）で形態素解析・トークナイズする
3. Word2Vecモデルを学習する
4. 孤独関連語の意味空間を可視化する

**分析の問い（Phase 1）**
- 「孤独」「孤立」「孤高」はどんな語と共起しているか？
- 時代（明治 / 大正 / 昭和）で意味空間は変わるか？
- Loneliness / Isolation / Solitude の方向ベクトルはデータから確認できるか？

---
> **注意**：青空文庫のテキストは著作権切れ作品が中心（概ね没後70年）。
> 明治〜昭和中期が主なカバー範囲になる。

---
## セッションが切れた場合の再開手順
1. 「ランタイム → すべてのセルを実行」で上から流し直す
2. ダウンロード済みの場合はparquetから読み込むのでスキップされる
3. Google Driveのデータは保持されるので再ダウンロード不要

## 0-A. Google Driveのマウント
データをGoogle Driveに保存することでセッションが切れてもデータが消えない。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# データ保存先をGoogle Driveに設定
DATA_DIR = Path('/content/drive/MyDrive/loneliness-research-data')
RAW_DIR = DATA_DIR / 'raw' / 'aozora'
PROCESSED_DIR = DATA_DIR / 'processed' / 'aozora'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Google Drive マウント完了')
print(f'データ保存先: {DATA_DIR}')

## 0-B. 環境セットアップ

In [ ]:
# ライブラリのインストール（Colab用）
!pip install fugashi unidic-lite gensim umap-learn japanize-matplotlib jaconv -q
print('インストール完了')

In [ ]:
import os
import re
import gc
import time
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns

import fugashi
import jaconv
from gensim.models import Word2Vec

# 再現性のためシードを固定
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('セットアップ完了')
print(f'  データ保存先: {RAW_DIR}')

---
## 1. 青空文庫からテキストを取得する

In [ ]:
# 青空文庫の作品リスト（CSV）を取得する
CATALOG_URL = 'https://www.aozora.gr.jp/index_pages/list_person_all_extended_utf8.zip'
CATALOG_ZIP = RAW_DIR / 'catalog.zip'
CATALOG_CSV = RAW_DIR / 'list_person_all_extended_utf8.csv'

if not CATALOG_CSV.exists():
    print('作品リストをダウンロード中...')
    urllib.request.urlretrieve(CATALOG_URL, CATALOG_ZIP)
    with zipfile.ZipFile(CATALOG_ZIP, 'r') as zf:
        zf.extractall(RAW_DIR)
    print('完了')
else:
    print('作品リストは取得済み')

df_catalog = pd.read_csv(CATALOG_CSV, encoding='utf-8')
print(f'総作品数: {len(df_catalog):,}')

In [ ]:
# 時代ラベルを付与する
def assign_era(birth_date_str):
    """生年月日から時代ラベルを返す"""
    try:
        year = int(str(birth_date_str)[:4])
        if year < 1868:
            return '江戸以前'
        elif year < 1900:
            return '明治'
        elif year < 1912:
            return '明治後期〜大正'
        elif year < 1926:
            return '大正〜昭和初期'
        elif year < 1945:
            return '昭和戦前'
        else:
            return '昭和戦後以降'
    except:
        return '不明'

df_catalog['era'] = df_catalog['生年月日'].apply(assign_era)
print(df_catalog['era'].value_counts())

In [ ]:
# カラム名を確定版で設定
TEXT_URL_COL = 'テキストファイルURL'
TITLE_COL = '作品名'
AUTHOR_COL = '姓'

# テキストURLがあり、江戸以前と不明を除いた作品に絞る
df_target = df_catalog[
    df_catalog[TEXT_URL_COL].notna() &
    (~df_catalog['era'].isin(['不明', '江戸以前']))
].copy()

print(f'対象作品数: {len(df_target):,}')
print(df_target['era'].value_counts())

In [ ]:
# テキストをダウンロード・クリーニングする関数
def download_aozora_text(url, save_path):
    try:
        tmp_zip = str(save_path) + '.zip'
        urllib.request.urlretrieve(url, tmp_zip)
        with zipfile.ZipFile(tmp_zip, 'r') as zf:
            txt_files = [f for f in zf.namelist() if f.endswith('.txt')]
            if not txt_files:
                return None
            with zf.open(txt_files[0]) as f:
                try:
                    content = f.read().decode('shift-jis')
                except:
                    content = f.read().decode('utf-8', errors='ignore')
        os.remove(tmp_zip)
        return content
    except Exception as e:
        print(f'  エラー: {e}')
        return None


def clean_aozora_text(text):
    if text is None:
        return ''
    if '-------' in text:
        text = text.split('-------')[-1]
    text = re.sub(r'《[^》]*》', '', text)
    text = re.sub(r'［＃[^］]*］', '', text)
    text = re.sub(r'｜', '', text)
    text = jaconv.z2h(text, kana=False, ascii=True, digit=True)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


print('関数定義完了')

In [ ]:
# 保存済みデータを読み込む（ダウンロード済みの場合はスキップ）
parquet_path = PROCESSED_DIR / 'corpus_full.parquet'

if parquet_path.exists():
    df_corpus_full = pd.read_parquet(parquet_path)
    print(f'保存済みデータを読み込みました: {len(df_corpus_full)}作品')
    print(df_corpus_full['era'].value_counts())
else:
    # 初回のみダウンロード実行（20〜30分かかる）
    ERA_LIMITS = {
        '明治':         200,
        '明治後期〜大正': 200,
        '大正〜昭和初期': 200,
        '昭和戦前':      74,
        '昭和戦後以降':  114,
    }
    df_full = (
        df_target
        .groupby('era', group_keys=False)
        .apply(lambda x: x.sample(
            min(ERA_LIMITS.get(x.name, len(x)), len(x)),
            random_state=RANDOM_SEED
        ), include_groups=False)
    )
    print(f'取得予定: {len(df_full)}作品')

    corpus_full = []
    errors = []
    for i, row in df_full.iterrows():
        url = row[TEXT_URL_COL]
        era = row['era']
        title = row[TITLE_COL]
        author = row[AUTHOR_COL]
        text = download_aozora_text(url, RAW_DIR / f'{i}.txt')
        cleaned = clean_aozora_text(text)
        if cleaned and len(cleaned) > 500:
            corpus_full.append({'text': cleaned, 'era': era, 'title': title, 'author': author})
        else:
            errors.append(title)
        time.sleep(0.3)

    print(f'取得完了: {len(corpus_full)}作品 / スキップ: {len(errors)}作品')
    df_corpus_full = pd.DataFrame(corpus_full)
    df_corpus_full.to_parquet(parquet_path, index=False)
    print('保存完了')

---
## 2. 形態素解析・トークナイズ

In [ ]:
# 形態素解析の設定
tagger = fugashi.Tagger()
TARGET_POS = {'名詞', '動詞', '形容詞', '副詞'}

def tokenize(text, pos_filter=TARGET_POS):
    sentences = []
    current = []
    for word in tagger(text):
        pos = word.feature.pos1
        surface = word.surface
        if surface in {'。', '！', '？', '\n\n'}:
            if len(current) >= 3:
                sentences.append(current)
            current = []
            continue
        if pos not in pos_filter:
            continue
        try:
            lemma = word.feature.lemma
            token = lemma if (lemma and lemma != '*') else surface
        except:
            token = surface
        if len(token) < 2 or token.isnumeric():
            continue
        current.append(token)
    if len(current) >= 3:
        sentences.append(current)
    return sentences

# 動作確認
test_text = '彼は孤独を感じていた。その孤立した日々の中で、ひとり静かに書き続けた。'
print('テスト結果:')
for sent in tokenize(test_text):
    print(' ', sent)

In [ ]:
# 全量コーパスで形態素解析（バッチ処理でメモリ節約）
print('トークナイズ開始...')
all_sentences_full = []
era_sentences_full = {}

BATCH_SIZE = 50  # 50作品ずつ処理

for batch_start in range(0, len(df_corpus_full), BATCH_SIZE):
    batch = df_corpus_full.iloc[batch_start:batch_start + BATCH_SIZE]
    for _, row in batch.iterrows():
        sentences = tokenize(row['text'])
        all_sentences_full.extend(sentences)
        era = row['era']
        if era not in era_sentences_full:
            era_sentences_full[era] = []
        era_sentences_full[era].extend(sentences)
    gc.collect()
    print(f'  {min(batch_start + BATCH_SIZE, len(df_corpus_full))} / {len(df_corpus_full)}作品処理済み')

print(f'\n総文数: {len(all_sentences_full):,}')
for era, sents in sorted(era_sentences_full.items()):
    print(f'  {era}: {len(sents):,}文')

---
## 3. Word2Vecモデルの学習

In [ ]:
W2V_PARAMS = dict(
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    epochs=10,
    workers=4,
    seed=RANDOM_SEED,
)

print('Word2Vec学習中...')
model = Word2Vec(sentences=all_sentences_full, **W2V_PARAMS)
print(f'語彙数: {len(model.wv):,}')
model.save(str(PROCESSED_DIR / 'w2v_aozora_all.model'))
print('モデル保存完了')

In [ ]:
# 孤独関連語の類似語を確認
SEED_WORDS = ['孤独', '孤立', '孤高', '寂しい', 'ひとり', '一人']

for word in SEED_WORDS:
    if word in model.wv:
        similar = model.wv.most_similar(word, topn=10)
        print(f'\n【{word}】に近い語:')
        for w, score in similar:
            print(f'  {w:10s}  {score:.3f}')
    else:
        print(f'\n【{word}】: 語彙になし')

---
## 4. 孤独語の意味空間を可視化する（UMAP）

In [ ]:
# 三分類のSeed語彙
LONELINESS_SEEDS = ['孤独', '寂しい', '悲しい', '虚しい', '空虚', '疎外', '取り残す', '心細い']
ISOLATION_SEEDS  = ['孤立', '断絶', '疎遠', '引きこもる', '排除', '遠ざかる', '孤立無援']
SOLITUDE_SEEDS   = ['孤高', '独り', '静かだ', '自由', '内省', '沈黙', '一人']
CONNECTION_SEEDS = ['友人', '家族', '絆', 'つながる', '仲間', '集う', '愛する']

def filter_vocab(words, model):
    return [w for w in words if w in model.wv]

loneliness_words = filter_vocab(LONELINESS_SEEDS, model)
isolation_words  = filter_vocab(ISOLATION_SEEDS, model)
solitude_words   = filter_vocab(SOLITUDE_SEEDS, model)
connection_words = filter_vocab(CONNECTION_SEEDS, model)

print('可視化対象語数:')
print(f'  Loneliness: {len(loneliness_words)} → {loneliness_words}')
print(f'  Isolation:  {len(isolation_words)} → {isolation_words}')
print(f'  Solitude:   {len(solitude_words)} → {solitude_words}')
print(f'  Connection: {len(connection_words)} → {connection_words}')

In [ ]:
import umap.umap_ as umap

viz_words = loneliness_words + isolation_words + solitude_words + connection_words
viz_labels = (
    ['Loneliness'] * len(loneliness_words) +
    ['Isolation']  * len(isolation_words) +
    ['Solitude']   * len(solitude_words) +
    ['Connection（比較）'] * len(connection_words)
)

vectors = np.array([model.wv[w] for w in viz_words])

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=min(10, len(viz_words)-1),
    random_state=RANDOM_SEED,
    metric='cosine'
)
embedding = reducer.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(12, 9))
COLORS  = {'Loneliness': '#E74C3C', 'Isolation': '#3498DB', 'Solitude': '#2ECC71', 'Connection（比較）': '#95A5A6'}
MARKERS = {'Loneliness': 'o',       'Isolation': 's',       'Solitude': '^',       'Connection（比較）': 'x'}

for label in set(viz_labels):
    idx = [i for i, l in enumerate(viz_labels) if l == label]
    ax.scatter(embedding[idx, 0], embedding[idx, 1],
               c=COLORS[label], marker=MARKERS[label], s=100, label=label, alpha=0.8)

for i, (word, label) in enumerate(zip(viz_words, viz_labels)):
    ax.annotate(word, (embedding[i, 0], embedding[i, 1]),
                fontsize=10, textcoords='offset points', xytext=(4, 4))

ax.legend(fontsize=11)
ax.set_title('孤独関連語の意味空間（青空文庫 Word2Vec + UMAP）', fontsize=14)
ax.set_xlabel('UMAP次元1')
ax.set_ylabel('UMAP次元2')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'umap_loneliness_words.png', dpi=150)
plt.show()
print('図を保存しました')

---
## 5. 方向ベクトルの試作（Loneliness / Isolation / Solitude 軸）

In [ ]:
from numpy.linalg import norm

def make_axis_vector(seed_words, model):
    vecs = [model.wv[w] for w in seed_words if w in model.wv]
    if not vecs:
        return None
    axis = np.mean(vecs, axis=0)
    return axis / norm(axis)

def score_on_axis(word, axis_vector, model):
    if word not in model.wv or axis_vector is None:
        return None
    wv = model.wv[word]
    return float(np.dot(wv, axis_vector) / (norm(wv) * norm(axis_vector)))

axis_loneliness = make_axis_vector(loneliness_words, model)
axis_isolation  = make_axis_vector(isolation_words, model)
axis_solitude   = make_axis_vector(solitude_words, model)

print('軸ベクトル作成完了')
print(f'  Loneliness軸: {len(loneliness_words)}語から')
print(f'  Isolation軸:  {len(isolation_words)}語から')
print(f'  Solitude軸:   {len(solitude_words)}語から')

In [ ]:
# 孤独周辺語を3軸でスコアリング
TARGET_WORDS_FOR_SCORING = [
    '孤独', '孤立', '孤高', '寂しい', '独り', '一人',
    '悲しい', '自由', '静かだ', '断絶', '友人', '家族', '愛する',
    '夜', '部屋', '窓', '海', '空'
]

rows = []
for word in TARGET_WORDS_FOR_SCORING:
    if word not in model.wv:
        continue
    rows.append({
        'word': word,
        'Loneliness': score_on_axis(word, axis_loneliness, model),
        'Isolation':  score_on_axis(word, axis_isolation, model),
        'Solitude':   score_on_axis(word, axis_solitude, model),
    })

df_scores = pd.DataFrame(rows).set_index('word')
print(df_scores.round(3).to_string())

In [ ]:
# ヒートマップで可視化
fig, ax = plt.subplots(figsize=(8, max(4, len(df_scores) * 0.5)))
sns.heatmap(df_scores, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-0.5, vmax=0.5, ax=ax)
ax.set_title('孤独語の三軸スコア（Loneliness / Isolation / Solitude）', fontsize=13)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'axis_scores_heatmap.png', dpi=150)
plt.show()

---
## 6. 時代別比較

In [ ]:
# 時代別モデルを学習
era_models = {}

for era, sents in sorted(era_sentences_full.items()):
    if len(sents) < 100:
        print(f'[{era}] データ不足でスキップ ({len(sents)}文)')
        continue
    print(f'[{era}] 学習中... ({len(sents):,}文)', end='')
    m = Word2Vec(sentences=sents, **W2V_PARAMS)
    era_models[era] = m
    print(f' 完了（語彙数: {len(m.wv):,}）')

print(f'\n学習済み時代: {list(era_models.keys())}')

In [ ]:
# 時代別の「孤独」類似語比較
FOCUS_WORD = '孤独'

for era, m in sorted(era_models.items()):
    if FOCUS_WORD in m.wv:
        similar = m.wv.most_similar(FOCUS_WORD, topn=10)
        print(f'\n【{era}】「{FOCUS_WORD}」に近い語:')
        for w, s in similar:
            print(f'  {w:12s} {s:.3f}')
    else:
        print(f'\n【{era}】「{FOCUS_WORD}」は語彙になし')

---
## メモ・気づき

（このセルに分析中の気づき・疑問点を随時記録する）

- 
- 

## 次のステップ

- [ ] BCCWJが届いたら同じパイプラインを適用
- [ ] Sentence-BERT に切り替えて文レベル分析
- [ ] 孤独語を含む文の抽出・クラスタリング